---
# Day-5
## Mini Project — Sentiment Classifier Pipeline

Now we put everything together. This pipeline applies every concept from today in sequence:

1. Clean the raw text
2. Tokenize
3. Remove stopwords (carefully — we keep negations)
4. Apply TF-IDF representation
5. Train a classifier
6. Evaluate and test on new sentences

This is a real, working NLP pipeline — the same structure used in production systems, just at a smaller scale.

In [1]:
import re
import string
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ── Dataset ────────────────────────────────────────────────────────────
texts = [
    "This product is absolutely amazing and I love it",
    "Terrible quality, completely disappointed with this purchase",
    "Best thing I have ever bought, highly recommend",
    "Waste of money, broke after one day",
    "Exceeded all my expectations, fantastic product",
    "Very poor quality, would not buy again",
    "Outstanding service and great product overall",
    "Awful experience from start to finish",
    "Absolutely love this, perfect in every way",
    "Not worth the price, very disappointing",
    "Great value for money, works perfectly",
    "Would not recommend this to anyone",
    "Incredible product, so happy with my purchase",
    "Complete garbage, avoid at all costs",
    "Really good quality and fast delivery",
    "Worst purchase I have ever made"
]
labels = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]  # 1 = Positive, 0 = Negative


# ── Preprocessing Function ─────────────────────────────────────────────
# We keep negation words (not, never, no) because they flip sentiment
negation_words = {"not", "never", "no", "nor", "neither"}
stop_words = set(stopwords.words('english')) - negation_words

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

processed_texts = [preprocess(t) for t in texts]


# ── Train / Test Split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    processed_texts, labels, test_size=0.25, random_state=42
)


# ── TF-IDF Vectorization ───────────────────────────────────────────────
vectorizer = TfidfVectorizer(ngram_range=(1, 2))  # unigrams + bigrams
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)


# ── Train Classifier ───────────────────────────────────────────────────
model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)


# ── Evaluate ───────────────────────────────────────────────────────────
y_pred = model.predict(X_test_vec)
print("Model Evaluation")
print("=" * 40)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


# ── Test on New Sentences ─────────────────────────────────────────────
new_reviews = [
    "I absolutely love this product",
    "This is not good at all",
    "Really disappointed with the quality",
    "Best purchase of the year without a doubt"
]

print("Predictions on New Reviews")
print("=" * 40)
for review in new_reviews:
    processed  = preprocess(review)
    vectorized = vectorizer.transform([processed])
    prediction = model.predict(vectorized)[0]
    label      = "Positive" if prediction == 1 else "Negative"
    confidence = model.predict_proba(vectorized).max()
    print(f"Review     : {review}")
    print(f"Prediction : {label}  (confidence: {confidence:.2%})")
    print()

Model Evaluation
Accuracy: 75.00%

              precision    recall  f1-score   support

    Negative       0.67      1.00      0.80         2
    Positive       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4

Predictions on New Reviews
Review     : I absolutely love this product
Prediction : Positive  (confidence: 58.30%)

Review     : This is not good at all
Prediction : Negative  (confidence: 56.90%)

Review     : Really disappointed with the quality
Prediction : Negative  (confidence: 50.36%)

Review     : Best purchase of the year without a doubt
Prediction : Positive  (confidence: 52.04%)

